# Actividad 6: llevando los LLM a producción

**Agente en producción:** https://fomento-socioeconomico.streamlit.app/  
**Repositorio:** https://github.com/engineercarloscogua/task6-productionLLM  
**Corpus documental:** https://drive.google.com/drive/folders/1soBQa1-s8zSwEri0gMOFH0E7BJgQ0Mt7?usp=sharing  

**Estudiantes:** Carlos Arnulfo Cogua Laverde, Michael Lan Alvarez y Marcela Gómez Garzón  
**Curso:** LLM

Este cuaderno reproduce de forma **independiente** la lógica principal del proyecto `task6-productionLLM`. No clona el repositorio ni importa sus módulos: permite probar el flujo en Google Colab y enlaza por separado la aplicación desplegada en Streamlit.

Al ejecutarlo se puede cargar uno o varios PDF, extraer su texto, dividirlo en fragmentos, construir una memoria vectorial TF-IDF, recuperar evidencia y formular una pregunta que será respondida por NVIDIA Nemotron mediante OpenRouter.

> Ejecuta las celdas de arriba hacia abajo. La memoria existe únicamente durante la sesión de Colab.

## 1. Fases y herramientas

| Fase | Herramienta | Función |
|---|---|---|
| Preparación | Python y `pip` | Instalar las dependencias mínimas. |
| Ingesta | `google.colab.files` | Cargar los PDF desde el computador. |
| Extracción | `pypdf` | Extraer texto y conservar archivo y página. |
| Chunking | Python | Crear fragmentos de 220 palabras con 40 de solapamiento. |
| Vectorización | TF-IDF local | Representar unigramas y bigramas como vectores dispersos. |
| Recuperación | Similitud coseno | Seleccionar los fragmentos más relacionados con la pregunta. |
| Generación | OpenRouter + NVIDIA Nemotron | Redactar una respuesta usando solamente la evidencia recuperada. |

El flujo implementado es: **PDF → texto por página → chunks → vectores TF-IDF → recuperación → prompt con contexto → respuesta**.

### Trazabilidad de la rúbrica

| Criterio | Evidencia en este cuaderno |
|---|---|
| Código estructurado y documentado | Fases 2 a 9, comentarios y docstrings |
| LangChain en producción | Cadena LCEL, tool, agente, RAG, memoria y enlace Streamlit |
| Rendimiento y escalabilidad | Medición real con concurrencias 1, 2 y 5 |
| Ética | Riesgos, sesgos, privacidad y mitigaciones |
| Resultados y conclusiones | Interpretación técnica y conclusiones finales |

## 2. Instalación

Solo se instalan las bibliotecas necesarias para leer PDF, llamar a OpenRouter y mostrar los resultados.

In [ ]:
%pip install -q pypdf requests pandas langchain==1.5.0 langgraph==1.2.22 langchain-openai==1.8.5

## 3. Configuración

Pega una clave temporal de OpenRouter en la constante `OPENROUTER_API_KEY`. Para esta demostración académica puede quedar escrita en la copia privada del cuaderno; no publiques una clave activa.

El modelo configurado es el mismo modelo Nemotron gratuito preferido por el proyecto local. Si OpenRouter cambia su disponibilidad, solo se debe actualizar `MODEL_ID`.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import math
import re
import unicodedata

import pandas as pd
import requests
from IPython.display import Markdown, display
from pypdf import PdfReader
from google.colab import files

# Clave desechable de OpenRouter para la demostración.
OPENROUTER_API_KEY = "PEGA_AQUI_TU_CLAVE_DE_OPENROUTER"
MODEL_ID = "nvidia/nemotron-3.5-lightning:free"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
RUN_LANGCHAIN_DEMOS = False  # Actívalo solo cuando necesites evidenciar LCEL y memoria.

# Valores equivalentes a src/pdf_knowledge.py del proyecto local.
CHUNK_WORDS = 220
OVERLAP_WORDS = 40
TOP_K = 3
PDF_DIRECTORY = Path("/content/pdfs_unitropico")
PDF_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Modelo:", MODEL_ID)
print("Demos LangChain con consumo de cuota:", RUN_LANGCHAIN_DEMOS)
print("Directorio temporal:", PDF_DIRECTORY)

## 4. Ingesta de documentos

La siguiente celda permite seleccionar varios PDF. Los archivos se guardan únicamente en el almacenamiento temporal de la sesión de Colab.

In [ ]:
def upload_pdfs(target_directory):
    """Carga PDF desde el equipo y devuelve sus rutas temporales en Colab."""
    uploaded = files.upload()
    pdf_paths = []
    for original_name, content in uploaded.items():
        safe_name = Path(original_name).name
        if not safe_name.lower().endswith(".pdf"):
            print(f"Omitido porque no es PDF: {safe_name}")
            continue
        destination = target_directory / safe_name
        destination.write_bytes(content)
        pdf_paths.append(destination)
    if not pdf_paths:
        raise ValueError("Debes cargar al menos un archivo PDF.")
    return pdf_paths

pdf_paths = upload_pdfs(PDF_DIRECTORY)
print(f"PDF cargados: {len(pdf_paths)}")
for pdf_path in pdf_paths:
    print(" -", pdf_path.name)

## 5. Extracción y chunking

`pypdf` extrae el texto seleccionable página por página. Cada página se divide en ventanas de 220 palabras con 40 palabras compartidas entre ventanas consecutivas. El solapamiento ayuda a no perder información ubicada en el límite de dos fragmentos.

Cada chunk conserva metadatos de archivo, página y número de fragmento para presentar fuentes verificables. Al igual que el proyecto local, esta demostración espera PDF con texto seleccionable; un PDF compuesto solo por imágenes requiere OCR previo.

In [ ]:
def split_text(text, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    """Normaliza el texto y lo divide en fragmentos solapados por palabras."""
    words = re.sub(r"\s+", " ", text).strip().split()
    if not words:
        return []
    if chunk_words < 1 or overlap_words < 0 or overlap_words >= chunk_words:
        raise ValueError("La configuración de fragmentos no es válida.")
    step = chunk_words - overlap_words
    return [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if words[start:start + chunk_words]
    ]

def extract_pdf_chunks(paths):
    """Extrae texto por página y crea chunks con metadatos de trazabilidad."""
    chunks = []
    errors = []
    pages_with_text = 0
    for path in paths:
        try:
            reader = PdfReader(str(path))
            if reader.is_encrypted and reader.decrypt("") == 0:
                raise ValueError("el archivo está cifrado")
            for page_number, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                page_chunks = split_text(page_text)
                if page_chunks:
                    pages_with_text += 1
                for chunk_number, content in enumerate(page_chunks, start=1):
                    chunks.append({
                        "content": content,
                        "source": path.name,
                        "page": page_number,
                        "chunk": chunk_number,
                    })
        except Exception as error:
            errors.append(f"{path.name}: {type(error).__name__}: {error}")
    return chunks, errors, pages_with_text

chunks, extraction_errors, pages_with_text = extract_pdf_chunks(pdf_paths)
if not chunks:
    raise ValueError("Los PDF no contienen texto extraíble. Aplica OCR y vuelve a cargarlos.")

print("Páginas con texto:", pages_with_text)
print("Fragmentos creados:", len(chunks))
print("Errores de extracción:", len(extraction_errors))
if extraction_errors:
    print(*extraction_errors, sep="\n- ")
display(pd.DataFrame(chunks)[["source", "page", "chunk"]].head(15))

## 6. Vectorización y memoria TF-IDF

La memoria sigue la implementación de `src/knowledge.py`: normaliza el texto, genera unigramas y bigramas, calcula pesos TF-IDF, conserva un vector disperso por chunk y compara la pregunta mediante similitud coseno.

La vectorización es local y determinista: no consume la API. OpenRouter se utiliza después de recuperar los fragmentos.

In [ ]:
class TfidfKnowledgeBase:
    """Mantiene chunks y vectores TF-IDF para recuperación local."""

    def __init__(self, items):
        """Construye el vocabulario, el IDF y un vector disperso por chunk."""
        self.items = list(items)
        tokenized = [self._terms(item["content"]) for item in self.items]
        document_frequency = Counter(
            term for terms in tokenized for term in set(terms)
        )
        total_documents = len(self.items)
        self.idf = {
            term: math.log((1 + total_documents) / (1 + frequency)) + 1
            for term, frequency in document_frequency.items()
        }
        self.document_vectors = [self._vector(terms) for terms in tokenized]

    @staticmethod
    def _terms(text):
        """Convierte texto en unigramas y bigramas normalizados sin tildes."""
        normalized = unicodedata.normalize("NFKD", text.casefold())
        ascii_text = "".join(
            char for char in normalized if not unicodedata.combining(char)
        )
        words = re.findall(r"[a-z0-9]+", ascii_text)
        bigrams = [f"{left}_{right}" for left, right in zip(words, words[1:])]
        return [*words, *bigrams]

    def _vector(self, terms):
        """Calcula un vector TF-IDF disperso para una lista de términos."""
        counts = Counter(terms)
        total = max(len(terms), 1)
        return {
            term: (count / total) * self.idf.get(term, 0.0)
            for term, count in counts.items()
            if term in self.idf
        }

    @staticmethod
    def _cosine(left, right):
        """Calcula la similitud coseno entre dos vectores dispersos."""
        common = set(left).intersection(right)
        numerator = sum(left[term] * right[term] for term in common)
        left_norm = math.sqrt(sum(value * value for value in left.values()))
        right_norm = math.sqrt(sum(value * value for value in right.values()))
        if not left_norm or not right_norm:
            return 0.0
        return numerator / (left_norm * right_norm)

    def search(self, query, k=TOP_K):
        """Devuelve los k chunks con mayor similitud positiva para una consulta."""
        if not query.strip():
            return []
        query_vector = self._vector(self._terms(query))
        scores = [self._cosine(query_vector, vector) for vector in self.document_vectors]
        ranked_indexes = sorted(
            range(len(scores)), key=lambda index: scores[index], reverse=True
        )
        results = []
        for index in ranked_indexes[:k]:
            if scores[index] <= 0:
                continue
            result = dict(self.items[index])
            result["score"] = float(scores[index])
            results.append(result)
        return results

knowledge_base = TfidfKnowledgeBase(chunks)
print(f"Memoria creada con {len(knowledge_base.document_vectors)} vectores TF-IDF.")
print(f"Vocabulario: {len(knowledge_base.idf)} términos y bigramas.")

## 7. Prueba del recuperador

Antes de llamar al modelo se comprueba qué evidencia recupera la memoria. Esto permite separar un problema de búsqueda de un problema de generación.

In [ ]:
def show_retrieval(results):
    """Muestra fuentes, páginas, similitudes y vistas previas de los resultados."""
    if not results:
        print("No se encontraron fragmentos relacionados.")
        return
    rows = [
        {
            "archivo": item["source"],
            "página": item["page"],
            "fragmento": item["chunk"],
            "similitud": round(item["score"], 4),
            "vista_previa": item["content"][:220] + "...",
        }
        for item in results
    ]
    display(pd.DataFrame(rows))

test_question = "¿Cuáles son los requisitos para pertenecer a Renta Joven?"
test_results = knowledge_base.search(test_question)
print("Pregunta de prueba:", test_question)
show_retrieval(test_results)

## 8. Generación aumentada con OpenRouter

La pregunta se vectoriza localmente y se recuperan hasta tres chunks. Después, Nemotron recibe la pregunta junto con esa evidencia. El prompt le indica que no use conocimiento externo, que reconozca cuando la información no sea suficiente y que cite archivo y página.

Para mantener el cuaderno enfocado en la demostración RAG, aquí se omiten autenticación, memoria conversacional, filtros y demás protecciones propias de la aplicación desplegada.

In [ ]:
SYSTEM_PROMPT = """Eres un asistente de consulta documental sobre programas socioeconómicos.
Responde en español usando solamente la evidencia recuperada de los PDF.
Contesta de manera directa y concreta en entre 3 y 5 oraciones completas.
La primera oración debe responder exactamente la pregunta; después agrega los datos relevantes.
Si la evidencia no contiene la respuesta, dilo claramente y no inventes información.
No dejes frases inconclusas. Incluye al final una línea de fuentes con archivo y página.
La información importante debe verificarse en el PDF original."""

def build_context(results):
    """Convierte los chunks recuperados en un bloque de evidencia para el modelo."""
    blocks = []
    for number, item in enumerate(results, start=1):
        blocks.append(
            f"[EVIDENCIA {number}]\n"
            f"Fuente: {item['source']}, página {item['page']}\n"
            f"Contenido: {item['content']}"
        )
    return "\n\n".join(blocks)

def response_is_incomplete(answer, finish_reason):
    """Detecta respuestas vacías, demasiado cortas o cortadas por límite de tokens."""
    clean_answer = (answer or "").strip()
    return (
        finish_reason == "length"
        or len(clean_answer.split()) < 20
        or (clean_answer and clean_answer[-1] not in ".!?)]}")
    )

class OpenRouterRateLimitError(RuntimeError):
    """Indica que la cuenta agotó la cuota informada por OpenRouter."""

def rate_limit_message(response):
    """Convierte un HTTP 429 en una explicación breve con fecha de reinicio."""
    reset_value = response.headers.get("X-RateLimit-Reset")
    try:
        error_headers = response.json().get("error", {}).get("metadata", {}).get("headers", {})
        reset_value = reset_value or error_headers.get("X-RateLimit-Reset")
    except (ValueError, AttributeError):
        pass
    reset_text = "el próximo reinicio diario de OpenRouter"
    if reset_value:
        try:
            timestamp = int(reset_value)
            if timestamp > 10_000_000_000:
                timestamp /= 1000
            reset_text = datetime.fromtimestamp(timestamp, timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        except (TypeError, ValueError, OSError):
            pass
    return (
        "Se agotó la cuota diaria de modelos gratuitos de esta cuenta de OpenRouter. "
        f"Podrás volver a consultar después de {reset_text}. "
        "Cambiar a otro modelo gratuito no evita este límite de la cuenta. "
        "También puedes usar una cuenta con créditos, pero no desactives ninguna validación de seguridad."
    )

def call_openrouter(question, context):
    """Consulta Nemotron una vez para proteger la cuota diaria gratuita."""
    if not OPENROUTER_API_KEY or OPENROUTER_API_KEY == "PEGA_AQUI_TU_CLAVE_DE_OPENROUTER":
        raise ValueError("Pega una clave válida de OpenRouter en la configuración.")
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Actividad 6 - RAG Unitrópico",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Pregunta: {question}\n\nEvidencia recuperada:\n{context}\n\n"
                    "Entrega una respuesta concreta, autosuficiente y formada por oraciones completas."
                ),
            },
        ],
        "temperature": 0,
        # Nemotron razona por defecto. Se desactiva para reservar la salida a la respuesta visible.
        "reasoning": {"enabled": False},
        "max_tokens": 1800,
    }
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=180)
    if response.status_code == 429:
        raise OpenRouterRateLimitError(rate_limit_message(response))
    if not response.ok:
        raise RuntimeError(
            f"OpenRouter respondió HTTP {response.status_code}: {response.text[:500]}"
        )
    data = response.json()
    choice = data["choices"][0]
    answer = (choice["message"].get("content") or "").strip()
    finish_reason = choice.get("finish_reason")
    if response_is_incomplete(answer, finish_reason):
        raise RuntimeError(
            "El modelo devolvió una respuesta incompleta. Vuelve a ejecutar la consulta "
            "manualmente si deseas gastar otra solicitud de la cuota."
        )
    return answer

def answer_question(question, k=TOP_K, show_sources=True):
    """Recupera evidencia, consulta Nemotron y devuelve respuesta y fuentes."""
    question = question.strip()
    if not question:
        raise ValueError("Escribe una pregunta.")
    results = knowledge_base.search(question, k=k)
    if show_sources:
        show_retrieval(results)
    if not results:
        return "No se encontraron fragmentos relacionados en los PDF cargados.", []
    answer = call_openrouter(question, build_context(results))
    return answer, results

print("Funciones de generación listas.")

## 9. Consulta interactiva

Ejecuta esta celda cada vez que quieras formular una nueva pregunta. Primero se muestran los fragmentos recuperados y luego la respuesta de Nemotron. Cada ejecución correcta consume una solicitud de la cuota de OpenRouter. Si la cuenta llega al límite diario, la celda mostrará la fecha de reinicio sin producir un traceback.

In [ ]:
question = input("Escribe una pregunta sobre los PDF: " ).strip()
try:
    answer, retrieved_sources = answer_question(question)
except OpenRouterRateLimitError as error:
    display(Markdown(f"### Cuota diaria de OpenRouter agotada\n\n{error}"))
else:
    print("\nRESPUESTA\n")
    print(answer)

## 10. Demostración autónoma de LangChain

Esta sección evidencia las funcionalidades solicitadas por la rúbrica sin importar el proyecto local. Utiliza el mismo índice TF-IDF construido anteriormente y lo conecta con una cadena LCEL, una tool documental, un agente y memoria conversacional de LangGraph.

La aplicación productiva implementa estos mismos conceptos con Streamlit, un modelo OpenRouter propio, `create_agent`, `consultar_documentos` e `InMemorySaver`.

In [ ]:
from langchain.agents import create_agent
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

def require_api_key():
    """Valida y devuelve la clave temporal configurada para la demostración."""
    if not OPENROUTER_API_KEY or OPENROUTER_API_KEY == "PEGA_AQUI_TU_CLAVE_DE_OPENROUTER":
        raise ValueError("Pega una clave válida de OpenRouter en la configuración.")
    return OPENROUTER_API_KEY

def is_rate_limit_error(error):
    """Reconoce el HTTP 429 envuelto por el cliente de LangChain."""
    message = str(error).casefold()
    return "429" in message or "rate limit" in message or "rate_limit" in message

@tool
def consultar_documentos(pregunta: str) -> str:
    """Recupera fragmentos relevantes de los PDF y conserva archivo y página."""
    results = knowledge_base.search(pregunta, k=TOP_K)
    if not results:
        return "No se encontraron documentos relacionados con la pregunta."
    return build_context(results)

langchain_model = ChatOpenAI(
    model=MODEL_ID,
    api_key=require_api_key(),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=1800,
    timeout=180,
    max_retries=0,  # No gastar cuota adicional al recibir HTTP 429.
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Actividad 6 - LangChain Unitrópico",
    },
    extra_body={"reasoning": {"enabled": False}},
)

print("Tool disponible:", consultar_documentos.name)
print("Modelo LangChain:", MODEL_ID)

### 10.1 Cadena LCEL

La cadena recibe una pregunta y el contexto recuperado, construye el prompt, llama a Nemotron y convierte la respuesta del modelo en texto.

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "Responde en español usando únicamente el contexto documental. "
        "Si el dato no aparece, indícalo y no inventes información.",
    ),
    ("human", "Pregunta: {question}\n\nContexto:\n{context}"),
])
rag_chain = RAG_PROMPT | langchain_model | StrOutputParser()

def run_rag_chain(question):
    """Recupera contexto y ejecuta la cadena LCEL documental."""
    context = consultar_documentos.invoke({"pregunta": question})
    return rag_chain.invoke({"question": question, "context": context})

if RUN_LANGCHAIN_DEMOS:
    chain_question = "¿Qué es el programa Renta Joven?"
    try:
        chain_answer = run_rag_chain(chain_question)
    except Exception as error:
        if not is_rate_limit_error(error):
            raise
        display(Markdown("### Cuota diaria de OpenRouter agotada\n\nLa cadena LCEL no se ejecutó. Espera al reinicio diario indicado por OpenRouter."))
    else:
        print("PREGUNTA DE LA CADENA:", chain_question)
        print("\nRESPUESTA DE LA CADENA:\n", chain_answer)
else:
    print("Cadena LCEL preparada. Activa RUN_LANGCHAIN_DEMOS para ejecutarla.")

### 10.2 Agente, tool y memoria conversacional

El agente recibe la obligación de consultar la tool antes de contestar. `InMemorySaver` conserva los mensajes asociados al mismo `thread_id`, lo que permite resolver preguntas de seguimiento sin mezclar sesiones diferentes.

In [ ]:
AGENT_PROMPT = """Eres un asistente documental sobre programas socioeconómicos.
Siempre utiliza consultar_documentos antes de responder.
Responde solamente con la evidencia recuperada y reconoce cuando el dato no aparece.
Contesta en español con oraciones completas y menciona archivo y página."""

agent = create_agent(
    model=langchain_model,
    tools=[consultar_documentos],
    system_prompt=AGENT_PROMPT,
    checkpointer=InMemorySaver(),
)

def run_memory_turn(question, thread_id):
    """Ejecuta un turno del agente y devuelve respuesta y tools utilizadas."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    tools_used = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            if call.get("name") not in tools_used:
                tools_used.append(call.get("name"))
    return result["messages"][-1].content, tools_used

memory_thread = "colab-rubrica-memoria"
memory_questions = [
    "¿Qué es el programa Renta Joven?",
    "¿Y cuáles son sus requisitos?",
]
memory_results = []
if RUN_LANGCHAIN_DEMOS:
    for memory_question in memory_questions:
        try:
            memory_answer, tools_used = run_memory_turn(memory_question, memory_thread)
        except Exception as error:
            if not is_rate_limit_error(error):
                raise
            display(Markdown("### Cuota diaria de OpenRouter agotada\n\nEl agente con memoria se detuvo sin realizar más reintentos."))
            break
        memory_results.append({
            "pregunta": memory_question,
            "respuesta": memory_answer,
            "tools": ", ".join(tools_used),
        })
        print("\nPREGUNTA:", memory_question)
        print("TOOLS:", tools_used)
        print("RESPUESTA:", memory_answer)
else:
    print("Agente y memoria preparados. Activa RUN_LANGCHAIN_DEMOS para ejecutarlos.")

## 11. Evidencia de producción

La demostración de Colab y el despliegue son artefactos independientes. El cuaderno permite inspeccionar y ejecutar la lógica; Streamlit ofrece la interfaz pública del agente.

- **Aplicación:** https://fomento-socioeconomico.streamlit.app/
- **Repositorio:** https://github.com/engineercarloscogua/task6-productionLLM
- **Framework de interfaz:** Streamlit.
- **Proveedor del modelo:** OpenRouter.
- **Modelo de la medición:** `nvidia/nemotron-3.5-lightning:free`.
- **Fecha de verificación y benchmark:** 20 de noviembre de 2026.

In [ ]:
PRODUCTION_URL = "https://fomento-socioeconomico.streamlit.app/"
try:
    production_response = requests.get(PRODUCTION_URL, timeout=30)
    production_check = {
        "url": PRODUCTION_URL,
        "http_status": production_response.status_code,
        "reachable": production_response.status_code == 200,
    }
except Exception as error:
    production_check = {
        "url": PRODUCTION_URL,
        "http_status": None,
        "reachable": False,
        "detail": f"{type(error).__name__}: {error}",
    }
print(production_check)

## 12. Rendimiento y cargas de trabajo

La medición real se ejecutó contra OpenRouter con tres solicitudes por nivel. Un resultado solo se contó como exitoso cuando el modelo fue invocado y el agente utilizó `consultar_documentos`. Los valores describen esta muestra y no garantizan capacidad sostenida.

In [ ]:
benchmark_results = [
    {"concurrencia": 1, "solicitudes": 3, "éxitos": 3, "fallos": 0, "promedio_s": 12.606, "p50_s": 10.130, "p95_s": 25.343},
    {"concurrencia": 2, "solicitudes": 3, "éxitos": 3, "fallos": 0, "promedio_s": 4.747, "p50_s": 5.746, "p95_s": 5.823},
    {"concurrencia": 5, "solicitudes": 3, "éxitos": 3, "fallos": 0, "promedio_s": 6.962, "p50_s": 8.502, "p95_s": 9.712},
]
benchmark_table = pd.DataFrame(benchmark_results)
display(benchmark_table)
print("Éxitos totales:", int(benchmark_table["éxitos"].sum()), "de", int(benchmark_table["solicitudes"].sum()))

### Interpretación del rendimiento

Las nueve solicitudes fueron correctas. La concurrencia 2 presentó el menor promedio. Al pasar de 2 a 5 solicitudes concurrentes la latencia promedio aumentó, aunque permaneció por debajo de la línea base secuencial, afectada por una primera respuesta de 25.343 segundos. La prueba demuestra funcionamiento con las tres cargas seleccionadas, pero una muestra de nueve consultas no demuestra escalabilidad sostenida ni disponibilidad futura.

## 13. Consideraciones éticas

**Alucinaciones.** El modelo puede presentar inferencias como hechos. Se mitiga restringiendo la respuesta al contexto recuperado, mostrando archivo y página, usando temperatura cero y solicitando verificación humana.

**Sesgo y cobertura.** El asistente solo conoce los PDF cargados. Los documentos pueden estar incompletos, desactualizados o representar de manera desigual a distintos grupos. TF-IDF favorece coincidencias literales y puede recuperar peor cuando el usuario emplea sinónimos. Se mitiga declarando el corpus, revisando su vigencia y reconociendo cuando no existe evidencia.

**Privacidad.** Los fragmentos recuperados se envían a OpenRouter. No deben cargarse expedientes ni datos sensibles sin autorización y controles institucionales. Las claves deben revocarse después de la demostración y no publicarse activas.

**Impacto.** El asistente informa, pero no decide beneficios, sanciones o admisiones. Una decisión económica o académica debe comprobarse en el documento oficial y contar con revisión humana.

**Disponibilidad.** Los modelos gratuitos tienen cuotas y disponibilidad variable. Los resultados del benchmark no constituyen un acuerdo de servicio.

## 14. Resultados, interpretación y conclusiones

### Resultados

El cuaderno carga varios PDF, conserva archivo y página, crea chunks solapados, construye una memoria TF-IDF y recupera evidencia mediante similitud coseno. La generación directa produce respuestas sustentadas. La sección de LangChain agrega una cadena LCEL, una tool, un agente y memoria conversacional. La aplicación pública presenta la misma finalidad mediante Streamlit.

### Interpretación

Separar recuperación y generación permite comprobar qué fragmentos sustentan la respuesta. La prueba de seguimiento demuestra el uso del mismo hilo conversacional. Las nueve consultas del benchmark utilizaron la herramienta documental y terminaron correctamente, aunque la variación de latencia muestra que un proveedor gratuito no ofrece tiempos constantes.

### Conclusiones

La solución integra agente, cadena, tool, RAG y memoria, y dispone de un despliegue público independiente del Colab. La evidencia confirma que el prototipo funciona con el corpus evaluado y con cargas concurrentes pequeñas. Sus límites principales son la recuperación léxica, la dependencia de documentos vigentes, la disponibilidad de OpenRouter y la naturaleza efímera de la memoria. Para un uso institucional se requerirían monitoreo continuo, evaluación con usuarios diversos, gobierno de datos y revisión humana.

## 15. Qué demuestra el entregable

- Los documentos se cargan y procesan dentro de Colab.
- La extracción conserva la trazabilidad por archivo y página.
- El chunking replica los valores del proyecto local.
- La vectorización TF-IDF y la similitud coseno funcionan sin depender de la API.
- La cadena LCEL, la tool, el agente y la memoria se ejecutan de manera autónoma.
- OpenRouter y Nemotron intervienen después de recuperar evidencia.
- La aplicación productiva está desplegada mediante Streamlit.
- El rendimiento se documenta con tres niveles de concurrencia.
- Se presentan riesgos éticos, interpretación y conclusiones.

### Limitaciones

La memoria se pierde cuando termina la sesión de Colab. Los PDF escaneados requieren OCR previo. TF-IDF depende de coincidencias léxicas y no captura todos los significados posibles. Los modelos gratuitos de OpenRouter pueden tener límites de solicitudes o disponibilidad variable.